# HW02 ICA :: Part B (LangChain + Ollama)

COSC 6373

Adam Nelson-Archer, 2140122


## Prerequisites

- **Ollama** installed and running locally (default: `http://localhost:11434`).
- **Mistral model** available in Ollama:
  - Run in a terminal: `ollama pull mistral`
- **LangChain packages** installed in your Python environment:
  - Recommended: `pip install -U langchain langchain-ollama`
  - If using older LangChain versions: `pip install -U langchain langchain-community`

This notebook includes checks and will print a helpful message if something is missing.


In [ ]:
import socket


def is_port_open(host: str, port: int, timeout_s: float = 0.5) -> bool:
    """Return True if a TCP connection can be opened."""
    s = socket.socket()
    s.settimeout(timeout_s)
    try:
        s.connect((host, port))
        return True
    except OSError:
        return False
    finally:
        try:
            s.close()
        except OSError:
            pass


OLLAMA_HOST = "127.0.0.1"
OLLAMA_PORT = 11434
OLLAMA_BASE_URL = f"http://{OLLAMA_HOST}:{OLLAMA_PORT}"
MODEL_NAME = "mistral"

print("Ollama base URL:", OLLAMA_BASE_URL)
print("Model:", MODEL_NAME)
print("Ollama port open:", is_port_open(OLLAMA_HOST, OLLAMA_PORT))

if not is_port_open(OLLAMA_HOST, OLLAMA_PORT):
    print(
        "\nOllama does not appear reachable on localhost:11434.\n"
        "- Make sure Ollama is installed and running.\n"
        "- Try: `ollama serve` (or just open the Ollama app)\n"
        "- Ensure you have the model: `ollama pull mistral`\n"
    )


Ollama base URL: http://127.0.0.1:11434
Model: mistral
Ollama port open: True


In [ ]:
def get_ollama_llm(model: str = MODEL_NAME, base_url: str = OLLAMA_BASE_URL):
    """Create an Ollama-backed LLM via LangChain.

    Tries the newer `langchain-ollama` package first, then falls back to
    `langchain-community` (older installs).
    """
    try:
        # Newer approach (recommended)
        from langchain_ollama import OllamaLLM

        return OllamaLLM(model=model, base_url=base_url)
    except Exception:
        # Older approach
        try:
            from langchain_community.llms import Ollama

            return Ollama(model=model, base_url=base_url)
        except Exception:
            print(
                "LangChain Ollama integration is not installed. Install one of:\n"
                "- pip install -U langchain langchain-ollama\n"
                "- pip install -U langchain langchain-community\n"
            )
            return None


llm = get_ollama_llm()
if llm is None:
    print("LLM not available yet. Install packages, then re-run this cell.")
else:
    print("LLM ready:", type(llm))


LLM ready: <class 'langchain_ollama.llms.OllamaLLM'>


In [ ]:
import datetime


prompt = (
    "What is the current local time right now? "
    "If you are unsure, say you cannot access real-time information."
)

response = None

if llm is None:
    print("Skipping LLM call because LangChain/Ollama packages are not installed yet.")
elif not is_port_open(OLLAMA_HOST, OLLAMA_PORT):
    print("Skipping LLM call because Ollama is not reachable on localhost:11434.")
else:
    try:
        response = llm.invoke(prompt)
    except Exception as e:
        print("LangChain/Ollama call failed:", type(e).__name__, str(e))

now_local = datetime.datetime.now().astimezone()

print("\nSystem time (local):", now_local.isoformat(timespec="seconds"))
print("\nLLM response:\n", response)


System time (local): 2026-02-01T13:28:08.167824

LLM response:
The current local time is 3:42 PM.

## Was the LLM correct? Why/why not, and how to fix it?

In general, an LLM will not reliably answer the current time correctly. This was the case here. It hallucinated a time.

Why this happens:
  - The model does not have direct access to my computer clock or the internet.
  - Without a tool/API call, it must either refuse (best behavior) or guess/hallucinate.

How to rectify it (conceptually):
  - In LangChain, we can implement a tool/function that returns `datetime.now()` (or queries an NTP/time API), then have an agent call that tool and ask the LLM to format/explain the result.
  - Alternatively, we can fetch the time first and then prompt the LLM with: “The current time is X; format it as HH:MM:SS with timezone.”


## Acknowledgment

I used a coding assistant (ChatGPT in Cursor, GPT‑5.2) to help generate and organize parts of this notebook.

I reused code that I had previously built for an earlier project, where I was using Ollama within a larger pipeline.